In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

source_path = Path("data/five_metal_compounds_permutation_features.csv")
selected_dir = Path("data/phi_mis_selected_feature_parts")
selected_dir.mkdir(parents=True, exist_ok=True)

if not source_path.is_file():
    raise FileNotFoundError(source_path.resolve())

total_selected = 0

for index, chunk in enumerate(
    tqdm(
        pd.read_csv(source_path, chunksize=20_000),
        desc="Selecting existing features",
    )
):
    numerator = (
        chunk["M_EN_mismatch"]
        + chunk["M_FIE_mismatch"]
    )
    denominator = chunk["M_AR_mismatch"].replace(0, np.nan)

    phi_mis = (numerator / denominator).replace(
        [np.inf, -np.inf],
        np.nan,
    )

    selected = chunk.loc[
        phi_mis.between(0, 5, inclusive="both")
    ]

    part_path = selected_dir / f"part_{index:04d}.csv"
    selected.to_csv(part_path, index=False)

    total_selected += len(selected)

if total_selected != 1_318_640:
    raise ValueError(
        f"Expected 1,318,640 selected compounds; "
        f"found {total_selected:,}."
    )

print("Selected feature rows:", f"{total_selected:,}")
print("Feature parts:", len(list(selected_dir.glob("part_*.csv"))))

Selecting existing features: 0it [00:00, ?it/s]

Selected feature rows: 1,318,640
Feature parts: 205


In [2]:
import joblib
import torch
import torch.nn as nn

FEATURE_DIR = Path("data/phi_mis_selected_feature_parts")
OUTPUT_DIR = Path("results/phi_mis_selected_predictions_parts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALER_PATH = Path("model/scaler_schemeD_pretrained.pkl")
MODEL_PATH = Path("model/mlp_schemeD_final_finetuned.pth")

CURRENTS = [50, 100, 200, 300, 400, 500]
BATCH_SIZE = 4096
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_files = sorted(FEATURE_DIR.glob("part_*.csv"))

if not feature_files:
    raise FileNotFoundError(
        f"No feature files found in {FEATURE_DIR.resolve()}"
    )

print("Number of feature files:", len(feature_files))
print("First feature file:", feature_files[0])
print("Device:", DEVICE)


class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims=None, dropout=0.02):
        super().__init__()

        if hidden_dims is None:
            hidden_dims = [256, 128, 64]

        feature_layers = []
        layer_dims = [input_dim] + hidden_dims

        for i in range(len(hidden_dims)):
            feature_layers.extend([
                nn.Linear(layer_dims[i], layer_dims[i + 1]),
                nn.BatchNorm1d(layer_dims[i + 1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])

        self.feature_extractor = nn.Sequential(*feature_layers)
        self.regressor = nn.Linear(hidden_dims[-1], 1)

    def forward(self, inputs):
        features = self.feature_extractor(inputs)
        return self.regressor(features)


scaler = joblib.load(SCALER_PATH)

first_columns = pd.read_csv(
    feature_files[0], nrows=0
).columns.tolist()

if "CurrentDensity" not in first_columns:
    if "composition_key" not in first_columns:
        raise ValueError(
            "Missing composition_key in the feature file."
        )

    first_columns.insert(
        first_columns.index("composition_key"),
        "CurrentDensity",
    )

feature_names = [
    column
    for column in first_columns
    if column not in [
        "compound",
        "composition_key",
        "overpotential",
    ]
]

expected_count = getattr(scaler, "n_features_in_", None)

if (
    expected_count is not None
    and len(feature_names) != expected_count
):
    raise ValueError(
        f"Feature count mismatch: found {len(feature_names)}, "
        f"but the scaler expects {expected_count}."
    )

expected_names = getattr(scaler, "feature_names_in_", None)

if (
    expected_names is not None
    and list(expected_names) != feature_names
):
    raise ValueError(
        "Feature names or order do not match the scaler."
    )

model = MLPRegressor(
    input_dim=len(feature_names)
).to(DEVICE)

model_state = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
)

model.load_state_dict(model_state)
model.eval()

print("Model input features:", feature_names)
print("Number of input features:", len(feature_names))
print("Model loaded successfully.")

Number of feature files: 205
First feature file: data\phi_mis_selected_feature_parts\part_0000.csv
Device: cpu
Model input features: ['metal_x_max', 'metal1_AN', 'metal2_AN', 'metal3_AN', 'metal4_AN', 'metal5_AN', 'M_AN', 'M_AN_mismatch', 'M_AR', 'M_AR_mismatch', 'M_FIE', 'M_FIE_mismatch', 'M_EA', 'M_EA_mismatch', 'M_EN', 'M_EN_mismatch', 'M_Nd', 'M_Nd_mismatch', 'M_Nv', 'M_Nv_mismatch', 'C_EN', 'N_metal', 's_mix_metal', 'anion_to_metal_ratio', 'CurrentDensity']
Number of input features: 25
Model loaded successfully.


In [6]:
def predict_one_current(feature_df, current):
    if feature_df.empty:
        return np.empty(0, dtype=np.float32)

    candidate_df = feature_df.copy()
    candidate_df["CurrentDensity"] = current

    missing_columns = [
        column
        for column in feature_names
        if column not in candidate_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing input features: {missing_columns}"
        )

    X = (
        candidate_df[feature_names]
        .to_numpy()
        .astype(np.float32)
    )

    if not np.isfinite(X).all():
        raise ValueError(
            f"Non-finite input values at {current} mA g^-1."
        )

    X_scaled = scaler.transform(X).astype(np.float32)
    predictions = np.empty(len(X_scaled), dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, len(X_scaled), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(X_scaled))
            batch = torch.from_numpy(
                X_scaled[start:end]
            ).to(DEVICE)

            predictions[start:end] = (
                model(batch)
                .cpu()
                .numpy()
                .ravel()
            )

    return predictions

In [7]:
for feature_path in tqdm(
    feature_files,
    desc="Re-predicting feature files",
):
    output_path = OUTPUT_DIR / feature_path.name

    feature_df = pd.read_csv(feature_path)

    result = pd.DataFrame({
        "compound": feature_df["compound"].astype(str),
        "composition_key": (
            feature_df["composition_key"].astype(str)
        ),
    })

    for current in CURRENTS:
        result[f"pred_{current}"] = predict_one_current(
            feature_df,
            current,
        )

    temporary_path = OUTPUT_DIR / f"{feature_path.stem}.tmp"

    result.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(output_path)

    print(
        f"Updated {output_path.name}: "
        f"{len(result):,} compositions",
        flush=True,
    )

print("Prediction completed.")
print("Output directory:", OUTPUT_DIR.resolve())
print(
    "Number of output files:",
    len(list(OUTPUT_DIR.glob("part_*.csv"))),
)

Re-predicting feature files:   0%|          | 0/205 [00:00<?, ?it/s]

Updated part_0000.csv: 1,471 compositions
Updated part_0001.csv: 1,446 compositions
Updated part_0002.csv: 1,492 compositions
Updated part_0003.csv: 1,446 compositions
Updated part_0004.csv: 1,518 compositions
Updated part_0005.csv: 1,470 compositions
Updated part_0006.csv: 1,450 compositions
Updated part_0007.csv: 1,426 compositions
Updated part_0008.csv: 1,456 compositions
Updated part_0009.csv: 1,069 compositions
Updated part_0010.csv: 527 compositions
Updated part_0011.csv: 533 compositions
Updated part_0012.csv: 516 compositions
Updated part_0013.csv: 550 compositions
Updated part_0014.csv: 538 compositions
Updated part_0015.csv: 535 compositions
Updated part_0016.csv: 522 compositions
Updated part_0017.csv: 581 compositions
Updated part_0018.csv: 523 compositions
Updated part_0019.csv: 505 compositions
Updated part_0020.csv: 499 compositions
Updated part_0021.csv: 498 compositions
Updated part_0022.csv: 501 compositions
Updated part_0023.csv: 506 compositions
Updated part_0024.cs

Updated part_0200.csv: 9,178 compositions
Updated part_0201.csv: 9,101 compositions
Updated part_0202.csv: 9,181 compositions
Updated part_0203.csv: 9,192 compositions
Updated part_0204.csv: 4,021 compositions
Prediction completed.
Output directory: D:\code\five-metal\results\phi_mis_selected_predictions_parts
Number of output files: 205


In [8]:
combined_path = Path(
    "results/phi_mis_selected_predictions_all_currents.csv"
)

prediction_files = sorted(
    OUTPUT_DIR.glob("part_*.csv")
)

if len(prediction_files) != len(feature_files):
    raise ValueError(
        f"Expected {len(feature_files)} prediction files, "
        f"but found {len(prediction_files)}."
    )

expected_columns = [
    "compound",
    "composition_key",
    "pred_50",
    "pred_100",
    "pred_200",
    "pred_300",
    "pred_400",
    "pred_500",
]

temporary_path = combined_path.with_suffix(".tmp")
total_rows = 0

with temporary_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as output:
    for index, feature_path in enumerate(
        tqdm(feature_files, desc="Combining predictions")
    ):
        prediction_path = OUTPUT_DIR / feature_path.name

        if not prediction_path.is_file():
            raise FileNotFoundError(
                f"Missing prediction file: {prediction_path}"
            )

        if (
            prediction_path.stat().st_mtime
            < feature_path.stat().st_mtime
        ):
            raise ValueError(
                "Prediction file is older than its feature file: "
                f"{prediction_path.name}"
            )

        part = pd.read_csv(prediction_path)

        if part.columns.tolist() != expected_columns:
            raise ValueError(
                f"Unexpected columns in {prediction_path.name}"
            )

        part.to_csv(
            output,
            index=False,
            header=(index == 0),
        )

        total_rows += len(part)

if total_rows != 1_318_640:
    raise ValueError(
        f"Expected 1,318,640 predictions, found {total_rows:,}. "
        "The temporary file was not published."
    )

temporary_path.replace(combined_path)

print("Combined file:", combined_path.resolve())
print("Total compositions:", f"{total_rows:,}")

Combining predictions:   0%|          | 0/205 [00:00<?, ?it/s]

Combined file: D:\code\five-metal\results\phi_mis_selected_predictions_all_currents.csv
Total compositions: 1,318,640
